# ATML PA3 - Task 2 runner (Kaggle, GPU T4 x2, Internet on)

Attach the output of the Task 2 data notebook as an input. Caches SigLIP features, then trains and evaluates the 196-token run on GPU 0 and the optional 49-token run on GPU 1 in parallel. To resume an interrupted session, also attach this notebook's previous output.

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/atml-pa3 && git clone -q https://github.com/sayim665/atml-pa3.git /kaggle/working/atml-pa3
!cd /kaggle/working/atml-pa3 && git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, shutil
os.environ["ATML2_DATA"] = "/kaggle/working/task2_data"
os.environ["ATML2_OUT"] = "/kaggle/working/task2_outputs"
os.environ["ATML2_RESULTS"] = "/kaggle/working/task2_results"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("/kaggle/working/task2_data", exist_ok=True)
manifest = sorted(glob.glob("/kaggle/input/**/task2_data/manifest.json", recursive=True))[0]
source = os.path.dirname(manifest)
print("data from", source)
shutil.copy(manifest, "/kaggle/working/task2_data/manifest.json")
if not os.path.exists("/kaggle/working/task2_data/images"):
    os.symlink(os.path.join(source, "images"), "/kaggle/working/task2_data/images")
for name in ("task2_outputs", "task2_results"):
    for previous in sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)):
        print("restoring", previous)
        shutil.copytree(previous, f"/kaggle/working/{name}", dirs_exist_ok=True)
os.chdir("/kaggle/working/atml-pa3")

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m task2.features

In [ ]:
%%bash
mkdir -p /kaggle/working/task2_outputs/logs
run() {
  CUDA_VISIBLE_DEVICES=$1 timeout 11h python -m task2.train --run $2 >> /kaggle/working/task2_outputs/logs/$2.log 2>&1 &&
  CUDA_VISIBLE_DEVICES=$1 python -m task2.evaluate --run $2 >> /kaggle/working/task2_outputs/logs/$2.log 2>&1
  echo "$2 finished with exit code $?"
}
run 0 vlm196 &
run 1 vlm49 &
wait

In [ ]:
!tail -n 5 /kaggle/working/task2_outputs/logs/*.log
!mkdir -p /kaggle/working/task2_results/features && cp /kaggle/working/task2_data/features/cache_info.json /kaggle/working/task2_results/features/
!for run in vlm196 vlm49; do mkdir -p /kaggle/working/task2_results/training/$run; cp /kaggle/working/task2_outputs/runs/$run/*.json* /kaggle/working/task2_results/training/$run/ 2>/dev/null; done
!for run in vlm196 vlm49; do [ -f /kaggle/working/task2_outputs/runs/$run/DONE ] && rm -rf /kaggle/working/task2_outputs/runs/$run/recovery; done
!rm -rf /kaggle/working/task2_data /kaggle/working/atml-pa3
!du -sh /kaggle/working/*